<a href="https://colab.research.google.com/github/naisyanabilapratiwi14-droid/Tugas-4-Sistem-temu-kembali/blob/main/240210500003_Naisya_Nabila_Pratiwi_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Tugas 4


**No. 1**



Tokenisasi, Stopwords Removal, dan Stemming

Studi Kasus: Sistem pencarian dokumen internal Fakultas Teknik UNM. Data berita/pengumuman
masih kotor (tanda baca, huruf kapital, stopwords, kata berimbuhan) dan perlu dibersihkan agar
siap digunakan untuk sistem temu kembali informasi (Information Retrieval).

Nama :_(Naisya Nabila Pratiwi)_
NIM  :_(240210500003)_

In [6]:
!pip install Sastrawi -q

import re # Import "re" untuk operasi regular expression (dipakai pada tahap cleaning)
import math # Import "math" untuk fungsi logaritma natural (ln), dipakai pada perhitungan IDF manual
import pandas as pd # Import "pandas" untuk menampilkan hasil dalam bentuk tabel (DataFrame)

from sklearn.feature_extraction.text import TfidfVectorizer # Import TfidfVectorizer dari scikit-learn untuk perhitungan TF-IDF menggunakan library

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory # Import factory pembuat stopword remover dari library Sastrawi (untuk Bahasa Indonesia)
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
stopwords_list = set(StopWordRemoverFactory().get_stop_words()) # Buat daftar stopwords Bahasa Indonesia dan simpan sebagai set (pencarian lebih cepat)
stemmer = StemmerFactory().create_stemmer() # Membuat stemmer Bahasa Indonesia

pd.set_option("display.max_columns", None) # Atur agar seluruh kolom pandas DataFrame ditampilkan (tidak terpotong)
pd.set_option("display.width", 120) # Atur lebar tampilan output pandas agar tabel lebih mudah dibaca

In [7]:
# List berisi 5 dokumen mentah (belum diproses) sesuai tema teknologi/komputer/pendidikan
documents = [
    # Dokumen 1 - bertema teknologi (kecerdasan buatan)
    "Perkembangan Teknologi Kecerdasan Buatan di Indonesia! Dalam 5 tahun terakhir, "
    "penggunaan kecerdasan buatan (AI) di berbagai sektor industri Indonesia meningkat pesat. "
    "Banyak perusahaan mulai mengadopsi AI untuk mengotomatisasi proses bisnis mereka, "
    "mulai dari layanan pelanggan hingga analisis data pelanggan.",

    # Dokumen 2 - bertema komputer (pengumuman laboratorium)
    "Pengumuman: Laboratorium Komputer Fakultas Teknik akan direnovasi mulai tanggal 12 Januari 2026. "
    "Seluruh mahasiswa diharapkan menggunakan laboratorium cadangan di Gedung B selama masa renovasi "
    "berlangsung selama kurang lebih 30 hari kerja.",

    # Dokumen 3 - bertema pendidikan (beasiswa)
    "Fakultas Teknik UNM kembali membuka program beasiswa unggulan bagi 100 mahasiswa berprestasi "
    "pada tahun ajaran 2026/2027. Pendaftaran beasiswa ini dapat dilakukan secara online melalui "
    "portal akademik kampus mulai minggu depan.",

    # Dokumen 4 - bertema teknologi/jaringan (infrastruktur kampus)
    "Jaringan internet kampus mengalami peningkatan kecepatan hingga 2x lipat setelah dilakukan "
    "upgrade infrastruktur fiber optik oleh tim IT Fakultas Teknik. Mahasiswa kini dapat mengakses "
    "e-learning dan jurnal digital dengan lebih cepat dan stabil.",

    # Dokumen 5 - bertema pendidikan/komputer (workshop)
    "Workshop Pemrograman Python untuk Pemula akan diselenggarakan oleh Himpunan Mahasiswa Teknik "
    "Komputer pada tanggal 20 Februari 2026. Workshop ini terbuka untuk seluruh mahasiswa dan tidak "
    "dipungut biaya pendaftaran sepeser pun!",
]

# Menampilkan setiap dokumen beserta nomornya, sebagai pengecekan awal dataset
for i, doc in enumerate(documents, 1):   # enumerate mulai dari 1 agar penomoran natural (1-5)
    print(f"Dokumen {i}:")               # cetak nomor dokumen
    print(doc)                           # cetak isi dokumen
    print("-" * 80)                      # cetak garis pemisah antar dokumen

Dokumen 1:
Perkembangan Teknologi Kecerdasan Buatan di Indonesia! Dalam 5 tahun terakhir, penggunaan kecerdasan buatan (AI) di berbagai sektor industri Indonesia meningkat pesat. Banyak perusahaan mulai mengadopsi AI untuk mengotomatisasi proses bisnis mereka, mulai dari layanan pelanggan hingga analisis data pelanggan.
--------------------------------------------------------------------------------
Dokumen 2:
Pengumuman: Laboratorium Komputer Fakultas Teknik akan direnovasi mulai tanggal 12 Januari 2026. Seluruh mahasiswa diharapkan menggunakan laboratorium cadangan di Gedung B selama masa renovasi berlangsung selama kurang lebih 30 hari kerja.
--------------------------------------------------------------------------------
Dokumen 3:
Fakultas Teknik UNM kembali membuka program beasiswa unggulan bagi 100 mahasiswa berprestasi pada tahun ajaran 2026/2027. Pendaftaran beasiswa ini dapat dilakukan secara online melalui portal akademik kampus mulai minggu depan.
--------------------------

In [8]:
#Fungsi preprocess_text(text)

def preprocess_text(text):
    """
    Melakukan preprocessing teks Bahasa Indonesia secara lengkap.

    Tahapan:
        1. Case folding      -> mengubah semua huruf menjadi huruf kecil
        2. Cleaning           -> menghapus angka, tanda baca, dan karakter khusus
        3. Tokenisasi         -> memecah teks menjadi token/kata
        4. Stopwords removal  -> menghapus kata-kata umum (Sastrawi)
        5. Stemming           -> mengubah kata berimbuhan ke kata dasar (Sastrawi)

    Parameters
    ----------
    text : str
        Teks mentah yang akan diproses.

    Returns
    -------
    dict berisi hasil tiap tahap penting, agar mudah dianalisis lebih lanjut.
    """

    # 1. CASE FOLDING
    text_lower = text.lower()                             # ubah seluruh huruf menjadi huruf kecil

    # 2. CLEANING
    # re.sub(pola, pengganti, teks) -> ganti semua karakter yang BUKAN huruf a-z atau spasi dengan spasi
    text_clean = re.sub(r"[^a-z\s]", " ", text_lower)      # hapus angka, tanda baca, simbol, dll
    text_clean = re.sub(r"\s+", " ", text_clean).strip()   # rapikan spasi ganda menjadi satu spasi, lalu trim

    # 3. TOKENISASI
    tokens = text_clean.split()                           # pecah string menjadi list kata berdasarkan spasi
    original_tokens = tokens.copy()                       # simpan salinan token SEBELUM stopwords/stemming

    # 4. STOPWORDS REMOVAL (Sastrawi)
    # list comprehension: ambil token yang TIDAK ada di dalam daftar stopwords
    tokens_no_stopwords = [t for t in tokens if t not in stopwords_list]

    # 5. STEMMING (Sastrawi)
    joined_for_stemming = " ".join(tokens_no_stopwords)   # gabungkan token menjadi satu string
    stemmed_text = stemmer.stem(joined_for_stemming)      # stemmer.stem() mengubah tiap kata ke kata dasar
    final_tokens = stemmed_text.split()                   # pecah kembali hasil stemming menjadi list token

    # Kembalikan seluruh hasil penting dalam bentuk dictionary
    return {
        "original_tokens": original_tokens,   # token sebelum stopwords removal & stemming
        "final_tokens": final_tokens,         # token setelah seluruh pipeline selesai
        "final_text": stemmed_text,           # hasil akhir dalam bentuk string
    }

In [9]:
#Penerapan pada 5 dokumen & perbandingan sebelum-sesudah

# Terapkan preprocess_text() ke seluruh dokumen, hasilnya disimpan dalam list "results"
results = [preprocess_text(doc) for doc in documents]     # list comprehension: proses tiap dokumen

# Cetak ringkasan jumlah token awal & akhir untuk seluruh dokumen (pengecekan cepat)
for i, res in enumerate(results, 1):                       # loop tiap hasil, nomor mulai dari 1
    n_before = len(res["original_tokens"])                 # jumlah token sebelum preprocessing penuh
    n_after = len(res["final_tokens"])                      # jumlah token sesudah preprocessing penuh
    print(f"Dokumen {i} -> token awal: {n_before}, token akhir: {n_after}")


def tampilkan_perbandingan(idx):
    """Menampilkan perbandingan lengkap (teks asli, tokenisasi, hasil akhir) untuk 1 dokumen."""
    doc = documents[idx]                                   # ambil teks asli dokumen ke-idx
    res = results[idx]                                      # ambil hasil preprocessing dokumen ke-idx

    print(f"{'='*90}\nDOKUMEN {idx + 1}\n{'='*90}")          # cetak header pemisah antar dokumen

    print("\n[1] TEKS ASLI:")                                # label bagian teks asli
    print(doc)                                              # cetak teks asli

    print(f"\n[2] SETELAH CASE FOLDING + CLEANING + TOKENISASI ({len(res['original_tokens'])} token):")
    print(res["original_tokens"])                           # cetak token sebelum stopwords/stemming

    print(f"\n[3] SETELAH STOPWORDS REMOVAL + STEMMING ({len(res['final_tokens'])} token):")
    print(res["final_tokens"])                              # cetak token setelah pipeline selesai

    print("\n[4] HASIL AKHIR (bentuk string):")
    print(res["final_text"])                                # cetak hasil akhir dalam bentuk kalimat
    print()                                                 # baris kosong sebagai pemisah


# Tampilkan perbandingan lengkap untuk 2 dokumen pertama (indeks 0 dan 1)
tampilkan_perbandingan(0)     # dokumen 1
tampilkan_perbandingan(1)     # dokumen 2

# Tampilkan ringkasan before -> after untuk SEMUA dokumen (termasuk dokumen 3, 4, 5)
print("=" * 90)
print("RINGKASAN SELURUH DOKUMEN (Sebelum -> Sesudah)")
print("=" * 90)
for i in range(len(documents)):                            # loop berdasarkan indeks 0..4
    print(f"Dokumen {i + 1}:")
    print("  Sebelum :", documents[i])                       # teks asli
    print("  Sesudah :", results[i]["final_text"])            # teks hasil preprocessing
    print()

Dokumen 1 -> token awal: 38, token akhir: 32
Dokumen 2 -> token awal: 28, token akhir: 26
Dokumen 3 -> token awal: 28, token akhir: 22
Dokumen 4 -> token awal: 34, token akhir: 28
Dokumen 5 -> token awal: 28, token akhir: 19
DOKUMEN 1

[1] TEKS ASLI:
Perkembangan Teknologi Kecerdasan Buatan di Indonesia! Dalam 5 tahun terakhir, penggunaan kecerdasan buatan (AI) di berbagai sektor industri Indonesia meningkat pesat. Banyak perusahaan mulai mengadopsi AI untuk mengotomatisasi proses bisnis mereka, mulai dari layanan pelanggan hingga analisis data pelanggan.

[2] SETELAH CASE FOLDING + CLEANING + TOKENISASI (38 token):
['perkembangan', 'teknologi', 'kecerdasan', 'buatan', 'di', 'indonesia', 'dalam', 'tahun', 'terakhir', 'penggunaan', 'kecerdasan', 'buatan', 'ai', 'di', 'berbagai', 'sektor', 'industri', 'indonesia', 'meningkat', 'pesat', 'banyak', 'perusahaan', 'mulai', 'mengadopsi', 'ai', 'untuk', 'mengotomatisasi', 'proses', 'bisnis', 'mereka', 'mulai', 'dari', 'layanan', 'pelanggan', 'h

In [10]:
# Statistik Jumlah Token dan Persentase Penguragan

# List kosong untuk menampung statistik tiap dokumen sebelum dimasukkan ke DataFrame
stats = []

for i, res in enumerate(results, 1):                        # loop tiap hasil, nomor mulai dari 1
    n_before = len(res["original_tokens"])                   # jumlah token sebelum preprocessing
    n_after = len(res["final_tokens"])                        # jumlah token sesudah preprocessing

    # Hitung persentase pengurangan token; dijaga agar tidak dibagi nol
    reduction_pct = (n_before - n_after) / n_before * 100 if n_before > 0 else 0

    # Tambahkan hasil perhitungan dokumen ini sebagai satu baris (dictionary) ke list "stats"
    stats.append({
        "Dokumen": f"Dokumen {i}",
        "Jumlah Token Sebelum": n_before,
        "Jumlah Token Sesudah": n_after,
        "Persentase Pengurangan (%)": round(reduction_pct, 2),   # dibulatkan 2 angka desimal
    })

# Ubah list of dict "stats" menjadi tabel pandas (DataFrame) agar mudah dibaca
df_stats = pd.DataFrame(stats)

# Hitung total token sebelum & sesudah dari seluruh dokumen (untuk baris ringkasan)
total_before = df_stats["Jumlah Token Sebelum"].sum()        # jumlahkan seluruh kolom "Sebelum"
total_after = df_stats["Jumlah Token Sesudah"].sum()          # jumlahkan seluruh kolom "Sesudah"
total_reduction = (total_before - total_after) / total_before * 100   # persentase pengurangan total

# Tambahkan baris baru berisi total/rata-rata ke bagian akhir tabel
df_stats.loc[len(df_stats)] = [
    "TOTAL / RATA-RATA",
    total_before,
    total_after,
    round(total_reduction, 2),
]

# Tampilkan tabel statistik akhir
df_stats

,Dokumen,Jumlah Token Sebelum,Jumlah Token Sesudah,Persentase Pengurangan (%)
0,Dokumen 1,38,32,15.79
1,Dokumen 2,28,26,7.14
2,Dokumen 3,28,22,21.43
3,Dokumen 4,34,28,17.65
4,Dokumen 5,28,19,32.14
5,TOTAL / RATA-RATA,156,127,18.59


## Analisis Singkat

Proses *preprocessing* (case folding, cleaning, tokenisasi, stopwords removal, dan stemming)
terbukti mampu menyederhanakan data teks secara signifikan, terlihat dari berkurangnya jumlah token
pada seluruh dokumen setelah pipeline dijalankan. Penghapusan tanda baca, angka, dan huruf kapital
membuat representasi teks lebih konsisten, sementara penghilangan *stopwords* membuang kata-kata
umum yang tidak memiliki daya pembeda (discriminative power) antar dokumen, dan proses stemming
menyatukan variasi kata berimbuhan (misalnya "menggunakan", "digunakan", "penggunaan") menjadi satu
kata dasar yang sama. Bagi sistem *temu kembali informasi* (Information Retrieval), hal ini sangat
bermanfaat karena mengurangi dimensi kosakata (vocabulary size), mempercepat proses pengindeksan dan
pencarian, serta meningkatkan kemungkinan dokumen yang relevan dapat ditemukan meskipun menggunakan
variasi kata yang berbeda dengan query pengguna, sehingga kualitas dan efisiensi sistem pencarian
menjadi lebih baik.